# CallGuard AI - Notebook 05: Fraud Risk & Indicator Detection Model

### Objective
Train and tune an end-to-end telephony fraud risk scoring model that calculates threat severity:
- Indicators: Law enforcement impersonation, OTP / 2FA credential theft, artificial urgency, payment diversion.
- Metrics: Zero tolerance for False Negatives (missed attacks) and minimization of False Positives (legitimate calls blocked).

In [ ]:
# Cell 2: Install dependencies & import libraries
!pip install -q scikit-learn pandas matplotlib seaborn joblib

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, f1_score

print("Fraud risk modeling dependencies loaded.")

In [ ]:
# Cell 3: Load fraud-labeled data
data_path = Path("ml/datasets/callguard/processed/cleaned_full.jsonl")
if not data_path.exists():
    data_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

df = pd.read_json(data_path, lines=True)
print(f"Loaded {len(df)} records. Risk level distribution:")
print(df["risk_level"].value_counts())

# Construct binary target: is_high_risk (high or critical risk)
df["is_fraud"] = df["risk_level"].isin(["high", "critical"]).astype(int)
print(f"\nBinary fraud label distribution:")
print(df["is_fraud"].value_counts())

In [ ]:
# Cell 4: Feature engineering for fraud signals
# Extract rule-based linguistic cues + TF-IDF n-grams
from ml.scripts.feature_engineering import extract_linguistic_features

ling_features_df = extract_linguistic_features(df["full_transcript"])
print("Linguistic features computed:", ling_features_df.columns.tolist())
display(ling_features_df.head(3))

# Fit TF-IDF on transcripts
tfidf_fraud = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english")
tfidf_sparse = tfidf_fraud.fit_transform(df["full_transcript"])

from scipy.sparse import hstack
X_composite = hstack([tfidf_sparse, ling_features_df.values])
y = df["is_fraud"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_composite, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Composite feature matrix: {X_composite.shape}")

In [ ]:
# Cell 5: Train fraud classifier
fraud_clf = LogisticRegression(class_weight={0: 1.0, 1: 3.0}, max_iter=1000, random_state=42)
fraud_clf.fit(X_train, y_train)
print("Fraud classifier trained with fraud class weighting.")

In [ ]:
# Cell 6: Evaluate (Focus on False Negatives and False Positives)
y_pred = fraud_clf.predict(X_test)
y_probs = fraud_clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=== Mission-Critical Fraud Metrics ===")
print(f"Total Test Calls       : {len(y_test)}")
print(f"True Negatives (Allowed) : {tn}")
print(f"False Positives (Blocked Genuine) : {fp} (FPR: {fp/(fp+tn):.2%})")
print(f"False Negatives (Missed Fraud)   : {fn} (FNR: {fn/(fn+tp):.2%})")
print(f"True Positives (Blocked Fraud)   : {tp}")

if fn > 0:
    print(f"CRITICAL ALERT: {fn} fraud calls were missed!")
else:
    print("SUCCESS: 0 fraud calls missed on the held-out test split.")

In [ ]:
# Cell 7: Threshold analysis — Precision-Recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], "b--", label="Precision (Genuine calls not blocked)")
plt.plot(thresholds, recalls[:-1], "g-", label="Recall (Fraud attacks caught)")
plt.title("Fraud Classifier Precision vs. Recall Across Decision Thresholds", fontsize=13, fontweight="bold")
plt.xlabel("Probability Decision Threshold")
plt.ylabel("Score")
plt.grid(True)
plt.axvline(0.35, color="red", linestyle=":", label="CallGuard Operational Threshold (0.35)")
plt.legend(loc="lower left")
plt.show()

In [ ]:
# Cell 8: Risk scoring function
def calculate_call_risk_score(probability: float, indicator_count: int) -> dict:
    """Produce structured 0-100 score and tier for backend orchestration."""
    base_score = probability * 80.0
    boost = min(20.0, indicator_count * 5.0)
    final_score = min(100.0, base_score + boost)
    
    if final_score < 25.0:
        tier = "low"
        action = "continue_ai"
    elif final_score < 55.0:
        tier = "medium"
        action = "continue_ai"
    elif final_score < 75.0:
        tier = "high"
        action = "transfer_human"
    else:
        tier = "critical"
        action = "block"
        
    return {
        "risk_score": round(final_score, 1),
        "risk_level": tier,
        "recommended_action": action
    }

# Test sample scoring
print("Sample High Risk Call Assessment:")
print(calculate_call_risk_score(probability=0.88, indicator_count=3))
print("\nSample Low Risk Call Assessment:")
print(calculate_call_risk_score(probability=0.08, indicator_count=0))

In [ ]:
# Cell 9: Save model
model_dir = Path("ml/models")
model_dir.mkdir(parents=True, exist_ok=True)

fraud_artifact = {
    "model": fraud_clf,
    "tfidf": tfidf_fraud,
    "threshold": 0.35
}

joblib.dump(fraud_artifact, model_dir / "fraud_risk_model_v1.0.0.joblib")

meta = {
    "name": "fraud_risk_model",
    "version": "1.0.0",
    "roc_auc": float(roc_auc_score(y_test, y_probs)),
    "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
    "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
    "operational_threshold": 0.35
}

with open(model_dir / "fraud_risk_model_v1.0.0.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved fraud model artifact and metadata card.")

# Cell 10: Fraud detection limitations + ethical considerations

### Limitations & Ethical Safeguards:
1. **False Positive Cost**: Blocking a genuine hospital notification or flight rescheduling damages user trust. We route moderate-risk calls to human agents rather than outright dropping them.
2. **Adversarial Adaptation**: Fraudsters adapt scripts when specific keywords are flagged (e.g. replacing 'OTP' with '6-digit authorization code'). Regular retraining is required.
3. **Dialect Fairness**: Linguistic markers (sentence length, exclamation) must not penalize non-native English speakers or regional speech patterns.